# NB 1.2 &mdash; El teu primer model, de principi a fi

**MP 5134** &mdash; UT1

*Versió amb els pingüins de l'arxipèlag Palmer.*

---
### Com funciona aquest notebook

Al revés del que esperes.

Primer executaràs un bloc de codi que entrena un model complet i et donarà un
número. **No entendràs res del que passa, i està bé així.** Després el
desmuntarem peça a peça.

Es fa així a propòsit. Si comencéssim per la teoria, arribaries al codi tres
sessions més tard sense saber per a què servia. Aquesta manera és més incòmoda
els primers deu minuts i molt més eficient la resta del curs.

## 1. El bloc complet

Executa la cel·la següent sense intentar entendre-la.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Còpia de les dades de palmerpenguins (Gorman et al., 2014) al repositori del mòdul
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"

df = pd.read_csv(URL_DADES)

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
data = df[FEATURES + ["body_mass_g"]].dropna()

X = data[FEATURES]
y = data["body_mass_g"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Puntuació del model: {model.score(X_test, y_test):.3f}")

Ja està. Acabes d'entrenar un model d'aprenentatge automàtic i de mesurar com de
bé funciona sobre dades que no havia vist mai.

Són quinze línies. La resta del notebook consisteix a entendre-les.

## 2. Pas a pas

### 2.1 Separar les entrades de la sortida

Tot problema supervisat es divideix sempre en dues peces:

- **X**: la taula de característiques, el que li donem al model.
- **y**: el vector objectiu, el que li demanem que endevini.

La convenció de posar `X` en majúscula i `y` en minúscula ve de les matemàtiques
i la trobaràs a tota la documentació i a tots els llibres: `X` és una taula de
dues dimensions i `y` és un vector d'una.

In [ ]:
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print()
print("Columnes de X:", list(X.columns))

Fixa't en tres decisions que hem pres en aquestes línies, i que no són innocents.

**`body_mass_g` no és a `X`.** Seria l'error més greu possible: donar-li al model
la resposta com a entrada. En diem **fuga d'informació** (*data leakage*) i és una
de les causes més freqüents de resultats espectaculars que després no funcionen.
Hi tornarem a la UT8 i a la UT10.

**`species`, `island` i `sex` tampoc hi són.** No perquè no serveixin &mdash;
al NB 1.1 vam veure que l'espècie explica molt bé la massa &mdash; sinó perquè
són text i encara no sabem convertir-les en números. Això és la UT3. Avui ens
limitem a les tres mesures numèriques, i val la pena que et quedis amb la idea
que **estem entrenant amb menys informació de la que tenim**.

**Hem fet `dropna()`.** Els onze pingüins amb valors absents que vam veure al
NB 1.1 no desapareixen sols: si no els traiem, la cel·la anterior hauria fallat.

### 2.2 Separar entrenament i prova

Aquesta és la idea més important de tot el mòdul, i la més fàcil de saltar-se.

Si entrenem el model amb totes les dades i després li preguntem sobre aquestes
mateixes dades, el resultat no ens diu res. És com corregir un examen a algú que
ha vist les respostes: no mesures si ha après, mesures si té memòria.

Per això apartem un tros de les dades **abans** d'entrenar, i no el tocarem fins
al moment d'avaluar.

In [ ]:
print(f"Mostres per entrenar: {len(X_train)}")
print(f"Mostres per provar:   {len(X_test)}  ({100 * len(X_test) / len(X):.0f} %)")

Dos paràmetres de `train_test_split` que veuràs cada dia:

`test_size=0.2` aparta el 20% de les dades per provar. El valor habitual està
entre el 10% i el 30%, i la **UT10** explicarà per què.

`random_state=42` fixa l'atzar. Sense això, cada execució faria una partició
diferent i obtindries un número lleugerament diferent cada vegada, cosa que fa
impossible comparar. El 42 no té cap significat tècnic; és una broma heretada
que s'ha convertit en costum.

Mira els números que acaben de sortir: entrenem amb 273 pingüins i n'avaluem 69.
Seixanta-nou. Apunta-t'ho, perquè a la secció 5 tornarem sobre aquesta xifra i no
serà per a bé.

### 2.3 L'estimador és un objecte

Aquí és on el que ja sabeu de programació orientada a objectes us dóna avantatge.

A scikit-learn, **un model és una classe**. Quan escrius `LinearRegression()`
estàs instanciant un objecte, igual que faries amb qualsevol altra classe.

In [ ]:
new_model = LinearRegression()

print("Tipus:", type(new_model))
print()
print("Paràmetres de configuració:")
for key, value in new_model.get_params().items():
    print(f"  {key} = {value}")

Aquest objecte acabat de crear **encara no sap res**. És un model buit, amb la
seva configuració però sense cap coneixement dels nostres pingüins.

El que li dóna contingut és el mètode `fit()`. I aquí hi ha la clau conceptual:
`fit()` **modifica l'estat intern de l'objecte**. Abans de cridar-lo, l'objecte
no té apresos els seus paràmetres; després, sí.

In [ ]:
print("Té coeficients abans de fit()?", hasattr(new_model, "coef_"))

new_model.fit(X_train, y_train)

print("Té coeficients després de fit()?", hasattr(new_model, "coef_"))
print()
for name, coef in zip(FEATURES, new_model.coef_):
    print(f"  {name:>18} : {coef: .2f}")

Aquests números són el que el model ha après. Cadascun diu quants grams puja o
baixa la predicció quan aquella mesura augmenta en un mil·límetre.

Val la pena llegir-los en veu alta: **cada mil·límetre d'aleta són uns 50 grams
de pingüí**. Les dues mesures del bec influeixen molt menys, i encara menys del
que sembla: el bec varia pocs mil·límetres d'un pingüí a l'altre, mentre que
l'aleta en varia seixanta. Té sentit, perquè l'aleta és una mesura de la mida
general de l'animal i el bec no.

Això ja apunta a un problema que et trobaràs de seguida: **els coeficients no es
poden comparar entre si si les variables estan en escales diferents**. És un dels
motius per escalar-les, i és contingut de la UT3.

Una convenció de scikit-learn que val la pena conèixer: **els atributs que acaben
amb guió baix (`coef_`, `intercept_`) són els que s'han après durant
l'entrenament**. Els que no en porten són configuració que has posat tu.

Aquesta interfície és idèntica per a tots els models de la biblioteca. Canviaràs
`LinearRegression` per `RandomForestRegressor` o per `SVC` i les tres crides
seran exactament les mateixes. Això és el que fa que puguem comparar algorismes
amb tanta facilitat durant tot el curs.

### 2.4 Predir i puntuar

Tres mètodes i ja tens el cicle complet:

| Mètode | Què fa |
|---|---|
| `fit(X, y)` | aprèn a partir dels exemples |
| `predict(X)` | dóna prediccions per a mostres noves |
| `score(X, y)` | mesura com de bé ho ha fet |

In [ ]:
predictions = model.predict(X_test)

comparison = pd.DataFrame({
    "real": y_test.values[:10].round(0),
    "predit": predictions[:10].round(0),
})
comparison["error"] = (comparison["predit"] - comparison["real"]).round(0)
comparison

Aquesta taula és més informativa que qualsevol mètrica. Mira els errors.

És important que t'hi acostumis des d'ara: **abans de mirar el número global,
mira uns quants casos concrets**. Els números globals amaguen problemes que els
exemples individuals ensenyen de seguida.

La majoria d'errors ronden els dos-cents o tres-cents grams. Sobre un pingüí de
quatre quilos, és un 5% o 7% de desviació. Que això sigui acceptable o no depèn
de per a què el vulguis, i aquesta pregunta &mdash; *quin error em puc permetre?*
&mdash; és sempre del client, no de l'algorisme.

In [ ]:
print(f"R2 sobre el conjunt de prova: {model.score(X_test, y_test):.3f}")

El número que dóna `score()` en un problema de regressió és el **R quadrat**.
Interpretació ràpida:

- **1.0** seria una predicció perfecta.
- **0.0** vol dir que el model no ho fa millor que dir sempre la mitjana.
- **negatiu** vol dir que ho fa pitjor que dir sempre la mitjana.

No et quedis amb la fórmula, que la veurem a la UT2. Queda't amb la idea: el R2
compara el teu model amb el model més ximple possible.

## 3. El model de referència

I aquí ve la pregunta que separa qui entén el que fa de qui només executa
cel·les: **aquest número és bo?**

No es pot respondre sense un punt de comparació. Construïm-lo.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

print(f"Referència (diu sempre la mitjana): {baseline.score(X_test, y_test):.3f}")
print(f"Regressió lineal:                   {model.score(X_test, y_test):.3f}")

`DummyRegressor` no aprèn res: diu sempre el mateix número, la massa mitjana de
tots els pingüins d'entrenament. Serveix per saber quant val la pena l'esforç.

Fixa't que la referència dóna pràcticament zero, i fins i tot una mica negatiu.
No és cap error: aquest model **és** la mitjana, així que per definició treu un R2
de zero sobre les dades amb què s'ha ajustat. Sobre el test surt lleugerament per
sota perquè la mitjana de l'entrenament no coincideix exactament amb la del test.

Aquest hàbit d'establir sempre una referència abans de celebrar un resultat
t'estalviarà disgustos. Un model amb un R2 de 0,85 sona molt bé fins que
descobreixes que dir sempre la mitjana ja en treu 0,80.

## 4. El mateix esquema, ara amb classificació

Canviem de problema: en lloc de la massa, volem saber **de quina espècie és el
pingüí**.

Observa que les línies clau són idèntiques. Només canvia la variable objectiu i
la classe del model.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

data_c = df[FEATURES + ["species"]].dropna()
Xc = data_c[FEATURES]
yc = data_c["species"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=42
)

classifier = DecisionTreeClassifier(max_depth=4, random_state=42)
classifier.fit(Xc_train, yc_train)

print(f"Encerts del classificador: {classifier.score(Xc_test, yc_test):.3f}")

Un percentatge d'encerts molt alt. Sembla un èxit.

Abans de creure-t'ho, aplica el que acabes d'aprendre: construeix la referència.

In [ ]:
from sklearn.dummy import DummyClassifier

ref_c = DummyClassifier(strategy="most_frequent")
ref_c.fit(Xc_train, yc_train)

print(f"Model que diu sempre 'Adelie': {ref_c.score(Xc_test, yc_test):.3f}")
print(f"El nostre arbre de decisió:    {classifier.score(Xc_test, yc_test):.3f}")

### Aquesta vegada sí

El model que no mira cap dada encerta la meitat de les vegades, perquè els Adelie
són gairebé la meitat de les mostres. El nostre arbre passa del 90%: de seixanta-
nou pingüins, n'encerta seixanta-cinc.

Aquí la diferència és tan gran que no hi ha discussió: **el model ha après alguna
cosa de veritat**. I no hauria de sorprendre'ns, perquè ja ho havíem vist dibuixat
al NB 1.1: al mapa de dispersió dels becs, les tres espècies apareixien gairebé
separades. L'arbre no ha fet res més que traçar les fronteres que tu ja havies
vist amb els ulls.

Val la pena que et quedis amb les dues cares de la mateixa moneda. El model de
referència no és un tràmit que sempre confirma que ho has fet bé: **és una
comparació, i de vegades la perds**. Hi ha problemes on el model que no mira cap
dada guanya el model entrenat, i quan això passa el que has descobert no és que
l'algorisme sigui dolent, sinó que la mètrica que has triat no serveix per a
aquell problema. Mateix codi, mateixa referència, conclusió oposada. En veuràs un
cas a la **UT4**.

La lliçó no és *"els models funcionen"* ni *"els models enganyen"*. És que **un
número tot sol no vol dir res fins que el compares amb alguna cosa**.

## 5. Una advertència sobre la mida

Tornem als seixanta-nou pingüins del conjunt de prova.

Amb un test tan petit, cada encert i cada error pesen molt: un sol pingüí mal
classificat són un punt i mig d'encerts. Vol dir que el número que hem obtingut
no és tan sòlid com sembla. Comprovem-ho: l'única cosa que canviarem és la
llavor de l'atzar de la partició.

In [ ]:
for llavor in [42, 7, 0, 1, 2]:
    Xr_train, Xr_test, yr_train, yr_test = train_test_split(
        X, y, test_size=0.2, random_state=llavor
    )
    r2 = LinearRegression().fit(Xr_train, yr_train).score(Xr_test, yr_test)

    Xs_train, Xs_test, ys_train, ys_test = train_test_split(
        Xc, yc, test_size=0.2, random_state=llavor
    )
    acc = DecisionTreeClassifier(max_depth=4, random_state=42).fit(
        Xs_train, ys_train).score(Xs_test, ys_test)

    print(f"random_state={llavor:>2} -> R2 = {r2:.3f}   encerts = {acc:.3f}")

Mira la columna del R2: entre la millor partició i la pitjor hi ha una diferència
d'una desena. **No hem canviat ni les dades ni el model**; només qui ha anat a
parar al conjunt de prova.

Això no és un defecte del nostre codi, és el preu de treballar amb 344 mostres.
Amb desenes de milers de files, el número gairebé no es mouria.

I aquí hi ha una distinció que val la pena entendre bé, perquè és subtil: **el
model no necessita més dades, la mesura sí**. Amb aquestes tres-centes mostres, la
regressió ja ha après tot el que podia aprendre; el que balla no és el que sap el
model, sinó la nostra estimació de com de bé ho fa.

Quan a la **UT10** et preguntis per què no n'hi ha prou amb partir les dades una
sola vegada i per què existeix la validació creuada, recorda aquesta taula. La
resposta és aquesta taula.

## 6. Exercici de lectura de codi

Torna al bloc de la secció 1 i respon per escrit, sense executar res:

**1.** Quina línia fa que el model aprengui?

**2.** Quina línia garanteix que l'avaluació sigui honesta?

**3.** Què passaria si `FEATURES` inclogués `"body_mass_g"`? Quin valor donaria
`score()` i per què això seria un error greu?

**4.** Si canvies `random_state=42` per `random_state=7`, canviarà el resultat?
Molt o poc? Respon primer i comprova-ho després amb la taula de la secció 5.

**5.** Reescriu el bloc canviant `LinearRegression` per `DecisionTreeRegressor`
(l'has d'importar de `sklearn.tree`). Quantes línies has hagut de tocar? Què et
diu això sobre el disseny de scikit-learn?

**6.** Al NB 1.1 vam veure que els Gentoo són molt més pesants que les altres dues
espècies. Si poguéssim afegir l'espècie com a característica, creus que el R2
pujaria gaire? Deixa la resposta escrita: la comprovarem a la UT3, quan sapiguem
convertir text en números.

## 7. Per al debat de classe

Pensa dos problemes del teu entorn (feina, aficions, el que sigui) que es
podrien plantejar com a aprenentatge automàtic.

Per a cadascun, defineix:

- Què seria una **mostra**?
- Quines serien les **característiques**?
- Quina seria la **variable objectiu**?
- És un problema de **regressió** o de **classificació**?
- I una que avui hem après a fer-nos: quin seria el **model de referència**? És a
  dir, què respondria algú que no mirés cap dada?

I una última, la més difícil: **d'on sortirien les dades?** En la pràctica
professional aquesta acostuma a ser la pregunta que decideix si un projecte és
viable o no. Els nostres pingüins van necessitar tres campanyes a l'Antàrtida.